# TTS Dataset Generation

Reads text prompts and writes a manifest with donor-synthesised audio, in the format
[docs/manifest.md](../docs/manifest.md) defines. This notebook is steps 1 and 2a of the chain
described in [docs/pipeline.md](../docs/pipeline.md): it turns a prompts source into the text
rows of a manifest, then synthesises each row with a single donor TTS voice. Voice conversion,
measurement and selection are later, separate notebooks
([tts_voice_convert.ipynb](tts_voice_convert.ipynb), [tts_measure.ipynb](tts_measure.ipynb),
[tts_select.ipynb](tts_select.ipynb)); this notebook only synthesises.

It never picks a donor from a language code. The donor comes from an admitted entry in
[registry/tts.yaml](../registry/tts.yaml) ([docs/registry.md](../docs/registry.md)) for
`TTS_LANG`, or from `DONOR_PLUGIN` and `DONOR_VOICE`, an explicit named candidate for
evaluation. A language with neither stops the notebook with a message naming both options.

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set the prompts source, the donor and every parameter via env vars |
| 1 | Load prompts | Read text from `PROMPTS_FILE`, `PROMPTS_DIR` or `PROMPTS_CSV`; refuse empty, bracketed or letterless rows |
| 2 | Donor synthesis | Resolve the donor from the registry or an explicit plugin, install it, synthesise every kept prompt |
| 3 | Post-processing | Resample to `TARGET_SR`, mono, silence trim, loudness normalisation, and an optional `audiosronnx` super-resolution stage |
| 4 | Manifest, sidecar and summary | Write `manifest.jsonl`, its `.meta.json`, `metadata.csv`, and print the counts |

### Output structure
```
OUTPUT_DIR/
  donor_raw/            # one raw donor WAV per kept prompt, before post-processing
  wavs/                 # the final, post-processed WAVs (manifest.jsonl points here)
  manifest.jsonl
  manifest.jsonl.meta.json
  metadata.csv          # LJSpeech-style "<id>|<text>", for convenience
```

### Inputs
- A prompts source: `PROMPTS_FILE` (one sentence per line, or piper-recording-studio
  `id<TAB>text`), `PROMPTS_DIR` (a folder of `.txt` files), or `PROMPTS_CSV` (a CSV/TSV with a
  text column). Set exactly one; with none set and `SMOKE=True`, a few built-in sentences are
  used instead.
- The donor: an admitted entry in `registry/tts.yaml` for `TTS_LANG`, or `DONOR_PLUGIN` and
  `DONOR_VOICE` named explicitly.

### Outputs
- `manifest.jsonl`: one row per synthesised prompt — `id`, `text`, `lang`, `audio`,
  `duration_s`, `sample_rate`, `source` (`docs/manifest.md`).
- `manifest.jsonl.meta.json`: the donor, whether it is registry-admitted, and the counts and
  refusal reasons from every stage.
- `metadata.csv`: the same rows as `<audio filename>|<text>`, for tools that read LJSpeech.

### Runtime
CPU, a few minutes for the smoke path (`SMOKE=True`, `N_SAMPLES` prompts). A full run's time
follows the donor plugin and the prompt count; no stage here needs a GPU.

### Hardware
CPU only. OVOS TTS plugins run on CPU; the optional `audiosronnx` super-resolution stage runs on
onnxruntime, CPU by default.

CI: executed on the smoke path (`SMOKE=True`; needs outbound network access to the smoke donor's
TTS service)

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

**Read more:**

- [OVOS TTS plugins](https://openvoiceos.github.io/ovos-technical-manual/320-tts_plugins/)
- [phoonnx](https://github.com/TigreGotico/phoonnx)
- [Making synthetic voices from scratch](https://openvoiceos.github.io/ovos-blogs/posts/2025-06-26-making-synthetic-voices-from-scratch)
- [synthetic_dataset_generator](https://github.com/TigreGotico/synthetic_dataset_generator), the pipeline this notebook continues

You can find `.txt` prompts here:
  - https://huggingface.co/datasets/fdemelo/phonetic-piper-recording-studio-prompts
  - https://huggingface.co/collections/TigreGotico/synthetic-tts-datasets
  - https://github.com/rhasspy/piper-recording-studio/tree/master/prompts


In [ ]:
# Parameters
SMOKE = False   # True: no network prompts/registry needed for text, but the smoke donor still calls out over the network
N_SAMPLES = 4   # row count of the built-in SMOKE prompts (mixes rows that pass and fail text validation)


> ⚠ **Donor synthesis needs outbound network access** for almost every OVOS TTS plugin (the
> smoke donor, `ovos-tts-plugin-edge-tts`, included): it calls a remote TTS service. A donor
> that runs fully offline (a local `phoonnx` voice, for example) does not need it, but the smoke
> path's own donor does.


---
## NGI0 Grant Attribution

> Developed by TigreGotico for OpenVoiceOS.

> This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

**NGI work item:** WP3 — TTS dataset pipeline, prompts and donor synthesis
**CI:** executed on the smoke path (`SMOKE=True`, needs outbound network to the smoke donor)

---


---
## 0 · Configuration

All parameters are set as environment variables via `os.environ.setdefault()`. Override any of
them from your shell, a `.env` file, or `%env` magic **before** running the config cell.

The pipeline, the manifest format and the language registry are described in
[docs/pipeline.md](../docs/pipeline.md), [docs/manifest.md](../docs/manifest.md) and
[docs/registry.md](../docs/registry.md).

**Edit the values below to match your setup, then run the cell.**


In [ ]:
import os
import re
from collections import Counter
from pathlib import Path


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


def _slug(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)


def _default_output_dir():
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return "/kaggle/working/tts_dataset_gen_out"
    if os.path.isdir("/content"):
        return "/content/tts_dataset_gen_out"
    return "./tts_dataset_gen_out"


# ──────────────────────────────────────────────
# GENERAL
# ──────────────────────────────────────────────
os.environ.setdefault("TTS_LANG",     "en-US")                 # BCP-47 tag with the variety, e.g. pt-PT
os.environ.setdefault("OUTPUT_DIR",   _default_output_dir())   # manifest, wavs/ and metadata.csv are written here
os.environ.setdefault("MANIFEST_NAME", "manifest.jsonl")
os.environ.setdefault("PROMPTS_SOURCE_NAME", "")                # source.name for ids; defaults to the prompts file/folder's own name

# ──────────────────────────────────────────────
# PROMPTS  (Stage 1) — set exactly one of these three
# ──────────────────────────────────────────────
os.environ.setdefault("PROMPTS_FILE", "prompts.txt")   # one sentence per line, or piper-recording-studio "id<TAB>text"
os.environ.setdefault("PROMPTS_DIR",  "")               # a folder of .txt files, each one sentence per line
os.environ.setdefault("PROMPTS_CSV",  "")               # a CSV/TSV with a text column (LJSpeech "id|text" also read)
os.environ.setdefault("CSV_TEXT_COL", "text")
os.environ.setdefault("CSV_SEP",      "|")

# ──────────────────────────────────────────────
# DONOR  (Stage 2, pipeline step 2a) — there is no default donor.
#   docs/registry.md admits a donor per TTS_LANG in registry/tts.yaml. When that language has
#   no admitted donor, DONOR_PLUGIN and DONOR_VOICE name an explicit candidate for evaluation
#   instead — never a language-to-engine guess. With neither, Stage 2 stops and names both
#   options. DONOR_PLUGIN is an OVOS TTS plugin entrypoint name (load_tts_plugin), e.g.
#   ovos-tts-plugin-edge-tts or ovos-tts-plugin-phoonnx.
# ──────────────────────────────────────────────
os.environ.setdefault("REGISTRY_PATH", "registry/tts.yaml")
os.environ.setdefault("DONOR_PLUGIN",  "")    # OVOS TTS plugin entrypoint name
os.environ.setdefault("DONOR_VOICE",   "")    # voice id for that plugin
os.environ.setdefault("DONOR_CONFIG",  "{}")  # JSON: extra plugin constructor options (explicit-donor mode only)
os.environ.setdefault("DONOR_PIP",     "")    # optional pip requirement for the package providing DONOR_PLUGIN; empty = already installed
os.environ.setdefault("DONOR_FORCE",   "0")   # 1 = re-synthesise a prompt even if its WAV already exists

# ──────────────────────────────────────────────
# POST-PROCESSING  (Stage 3)
# ──────────────────────────────────────────────
os.environ.setdefault("TARGET_SR",    "24000")   # final sample rate
os.environ.setdefault("TRIM_SILENCE", "1")       # 1 = trim leading/trailing silence
os.environ.setdefault("TRIM_TOP_DB",  "30")      # dB threshold for silence trimming
os.environ.setdefault("TARGET_DBFS",  "-20.0")   # loudness normalisation target

# Optional super-resolution (audiosronnx). Empty SR_ENGINE = off, plain resample only.
os.environ.setdefault("SR_ENGINE",      "")      # audiosronnx load_sr() engine alias; audiosronnx.available_models() lists them
os.environ.setdefault("SR_CACHE_DIR",   "")      # where audiosronnx caches downloaded weights; empty = its own default
os.environ.setdefault("SR_REVISION",    "")      # pin a Hugging Face revision for the SR_ENGINE weights
os.environ.setdefault("ONNX_PROVIDERS", "auto")  # onnxruntime execution providers for SR_ENGINE: comma list, or auto

print("Configuration loaded.")
print(f"  TTS_LANG   : {_env('TTS_LANG')}")
print(f"  OUTPUT_DIR : {_env('OUTPUT_DIR')}")
print(f"  Donor      : registry/tts.yaml first, else DONOR_PLUGIN={_env('DONOR_PLUGIN') or '(unset)'} "
      f"/ DONOR_VOICE={_env('DONOR_VOICE') or '(unset)'}")
print(f"  Super-res  : {_env('SR_ENGINE') or ('off (plain resample to ' + _env('TARGET_SR') + ' Hz)')}")


---
## 1 · Load Prompts

Reads text from whichever prompts source is configured and refuses a row whose text is empty,
carries a bracketed annotation (`[...]` or `<...>`), or has no letter — the same three rules
[tts_corpus_ingest.ipynb](tts_corpus_ingest.ipynb) applies to a corpus. A refusal is counted by
its reason; nothing here rewrites a prompt's text.

| Variable | Meaning |
|---|---|
| `PROMPTS_FILE` | One sentence per line, or piper-recording-studio `id<TAB>text` |
| `PROMPTS_DIR` | A folder of `.txt` files, each one sentence per line |
| `PROMPTS_CSV`, `CSV_TEXT_COL`, `CSV_SEP` | A CSV/TSV and its text column |


In [ ]:
def _clean_line(text):
    return re.sub(r"\s+", " ", text.strip())


def _read_lines_file(path):
    """One sentence per line, or piper-recording-studio 'id<TAB>text'."""
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if "\t" in line:
                line = line.split("\t", 1)[1]
            lines.append(_clean_line(line))
    return lines


def _read_lines_dir(folder):
    lines = []
    for txt in sorted(Path(folder).rglob("*.txt")):
        lines.extend(_read_lines_file(txt))
    return lines


def _read_lines_csv(path, text_col, sep):
    import csv
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        sample = f.read(4096)
        f.seek(0)
        has_header = bool(sample.strip()) and csv.Sniffer().has_header(sample)
        reader = csv.reader(f, delimiter=sep)
        if has_header:
            header = [h.strip().lower() for h in next(reader)]
            col_idx = header.index(text_col.lower()) if text_col.lower() in header else min(1, len(header) - 1)
        else:
            col_idx = 1 if sep == "|" else 0
        for row in reader:
            if len(row) > col_idx:
                lines.append(_clean_line(row[col_idx]))
    return lines


_BRACKET_RE = re.compile(r"\[[^\]]*\]|<[^>]*>")


def _text_refusal_reason(text):
    if not text or not text.strip():
        return "empty"
    if _BRACKET_RE.search(text):
        return "bracketed_annotation"
    if not any(ch.isalpha() for ch in text):
        return "no_letter"
    return None


def _smoke_prompt_lines():
    return [
        "the quick brown fox jumps over the lazy dog",
        "open voice os is a free and open assistant",
        "[inaudible]",           # refused: bracketed annotation
        "12345",                 # refused: no letter
    ]


csv_path, dir_path, file_path = _env("PROMPTS_CSV"), _env("PROMPTS_DIR"), _env("PROMPTS_FILE")

if csv_path and os.path.exists(csv_path):
    raw_lines = _read_lines_csv(csv_path, _env("CSV_TEXT_COL", "text"), _env("CSV_SEP", "|"))
    prompts_source_kind = "csv"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or Path(csv_path).stem
    prompts_source_path = csv_path
elif dir_path and os.path.isdir(dir_path):
    raw_lines = _read_lines_dir(dir_path)
    prompts_source_kind = "dir"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or Path(dir_path).name
    prompts_source_path = dir_path
elif file_path and os.path.exists(file_path):
    raw_lines = _read_lines_file(file_path)
    prompts_source_kind = "file"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or Path(file_path).stem
    prompts_source_path = file_path
elif SMOKE:
    raw_lines = _smoke_prompt_lines()[:max(int(N_SAMPLES), 1)]
    prompts_source_kind = "smoke"
    prompts_source_name = _env("PROMPTS_SOURCE_NAME") or "smoke"
    prompts_source_path = None
    print("No prompt source configured — using the built-in smoke prompts.")
else:
    raise RuntimeError(
        "No prompt source configured. Set one of PROMPTS_FILE, PROMPTS_DIR or PROMPTS_CSV."
    )

reason_counts = Counter()
PROMPT_ROWS = []
for text in raw_lines:
    reason = _text_refusal_reason(text)
    if reason:
        reason_counts[reason] += 1
        continue
    PROMPT_ROWS.append({"id": f"{prompts_source_name}:{len(PROMPT_ROWS):06d}", "text": text})
    reason_counts["passed"] += 1

print(f"Prompt lines read : {len(raw_lines)}")
print("Text validation:")
for reason in ("empty", "bracketed_annotation", "no_letter", "passed"):
    print(f"  {reason:22s} {reason_counts.get(reason, 0)}")
if not PROMPT_ROWS:
    raise RuntimeError("Every prompt was refused, or none were loaded; nothing to synthesise.")


---
## 2 · Donor Synthesis

Resolves the donor — [registry/tts.yaml](../registry/tts.yaml) first, an explicit
`DONOR_PLUGIN`/`DONOR_VOICE` otherwise — loads it through
`ovos_plugin_manager.tts.load_tts_plugin`, and synthesises every prompt that passed Stage 1.
There is no default donor and no language-to-engine mapping: a donor not admitted in the
registry is printed and recorded as a **candidate for evaluation**, never as a decided choice.

A row whose WAV already exists (from a previous run) is skipped unless `DONOR_FORCE=1`.


In [ ]:
import json
import shlex

%pip install --quiet --pre "PyYAML>=6.0" "ovos-plugin-manager>=2.11.6a1" "nest-asyncio>=1.6"

import nest_asyncio
nest_asyncio.apply()

import yaml


def _resolve_registry_path(registry_path):
    """The configured path, or the same name one directory up: a Jupyter user's
    default working directory is notebooks/, one level below the repository root
    where registry/tts.yaml actually lives."""
    for candidate in (Path(registry_path), Path("..") / registry_path):
        if candidate.is_file():
            return candidate
    return None


def _registry_donor(registry_path, lang):
    """The admitted donor for `lang` in registry/tts.yaml (docs/registry.md), or None."""
    p = _resolve_registry_path(registry_path)
    if p is None:
        print(f"  Registry not found at {registry_path!r} (also tried '../{registry_path}') "
              f"— no admitted donor consulted for {lang!r}.")
        return None
    data = yaml.safe_load(p.read_text(encoding="utf-8")) or {}
    for entry in data.get("entries") or []:
        if entry.get("lang") == lang:
            donor = entry.get("donor")
            if donor and donor.get("plugin"):
                return {"plugin": donor["plugin"], "voice": donor.get("voice") or ""}
            return None
    return None


registry_donor = _registry_donor(_env("REGISTRY_PATH", "registry/tts.yaml"), _env("TTS_LANG"))

if registry_donor:
    DONOR_PLUGIN_ID = registry_donor["plugin"]
    DONOR_VOICE_ID = registry_donor["voice"]
    DONOR_EXTRA_CONFIG = {}
    DONOR_ADMITTED = True
    DONOR_NOTE = f"registry/tts.yaml admits {DONOR_PLUGIN_ID!r} for {_env('TTS_LANG')!r}"
elif SMOKE and not _env("DONOR_PLUGIN") and not _env("DONOR_VOICE"):
    DONOR_PLUGIN_ID = "ovos-tts-plugin-edge-tts"
    DONOR_VOICE_ID = "en-US-JennyNeural"
    DONOR_EXTRA_CONFIG = {}
    DONOR_ADMITTED = False
    DONOR_NOTE = (f"SMOKE=True and {_env('TTS_LANG')!r} has no registry/tts.yaml entry: using the "
                  f"smoke donor {DONOR_PLUGIN_ID!r} ({DONOR_VOICE_ID}) — candidate for evaluation, "
                  f"not registry-admitted")
    if not _env("DONOR_PIP"):
        # Cell 5's os.environ.setdefault("DONOR_PIP", "") already claimed the key,
        # so setdefault here would be a no-op: assign directly, only when still empty.
        os.environ["DONOR_PIP"] = "ovos-tts-plugin-edge-tts>=0.3.2a2"
elif _env("DONOR_PLUGIN") and _env("DONOR_VOICE"):
    DONOR_PLUGIN_ID = _env("DONOR_PLUGIN")
    DONOR_VOICE_ID = _env("DONOR_VOICE")
    DONOR_EXTRA_CONFIG = json.loads(_env("DONOR_CONFIG", "{}"))
    DONOR_ADMITTED = False
    DONOR_NOTE = (f"{DONOR_PLUGIN_ID!r} is not admitted in registry/tts.yaml for "
                  f"{_env('TTS_LANG')!r} — candidate for evaluation")
else:
    raise RuntimeError(
        f"No donor is configured for TTS_LANG={_env('TTS_LANG')!r}. Either add an admitted entry "
        f"to registry/tts.yaml (docs/registry.md) with a 'donor' for this language, or set "
        f"DONOR_PLUGIN and DONOR_VOICE explicitly (docs/pipeline.md, step 2a) to name a candidate "
        f"for evaluation."
    )

print(f"Donor: {DONOR_NOTE}")

donor_pip = _env("DONOR_PIP").strip()
if donor_pip:
    get_ipython().run_line_magic("pip", f"install --quiet --pre {shlex.quote(donor_pip)}")
else:
    print(f"DONOR_PIP not set — assuming the package providing {DONOR_PLUGIN_ID!r} is already installed.")


In [ ]:
import sys
from tqdm import tqdm

from ovos_plugin_manager.tts import load_tts_plugin


def _plugin_package_version(plugin_cls, plugin_name):
    """The installed package version behind a plugin class. The entrypoint name (used to load
    the plugin) is not always the PyPI distribution name (phoonnx's OVOS entrypoint is
    ovos-tts-plugin-phoonnx, its distribution is phoonnx), so the class's own top-level module
    is tried first."""
    import importlib.metadata
    for candidate in (plugin_cls.__module__.split(".")[0], plugin_name, plugin_name.replace("_", "-")):
        try:
            return importlib.metadata.version(candidate)
        except importlib.metadata.PackageNotFoundError:
            continue
    return None


donor_cls = load_tts_plugin(DONOR_PLUGIN_ID)
if donor_cls is None:
    raise RuntimeError(
        f"OVOS TTS plugin {DONOR_PLUGIN_ID!r} is not installed. Set DONOR_PIP to the "
        f"package that provides it, or install it yourself before running this cell."
    )
donor_config = dict(DONOR_EXTRA_CONFIG)
donor_config.setdefault("lang", _env("TTS_LANG"))
if DONOR_VOICE_ID:
    donor_config.setdefault("voice", DONOR_VOICE_ID)
donor_plugin = donor_cls(config=donor_config)
DONOR_VERSION = _plugin_package_version(donor_cls, DONOR_PLUGIN_ID)
print(f"Loaded donor plugin {DONOR_PLUGIN_ID!r} (package version {DONOR_VERSION or 'unknown'})")

raw_dir = Path(_env("OUTPUT_DIR")) / "donor_raw"
raw_dir.mkdir(parents=True, exist_ok=True)
force = _flag("DONOR_FORCE")

SYNTH_ROWS = []
n_synth_failed = 0
for row in tqdm(PROMPT_ROWS, desc="Donor synthesis", unit="utt", file=sys.stdout):
    raw_path = raw_dir / f"{_slug(row['id'])}.wav"
    if raw_path.is_file() and raw_path.stat().st_size > 0 and not force:
        SYNTH_ROWS.append({**row, "raw_audio": str(raw_path)})
        continue
    try:
        donor_plugin.get_tts(row["text"], str(raw_path), lang=_env("TTS_LANG"), voice=DONOR_VOICE_ID or None)
        SYNTH_ROWS.append({**row, "raw_audio": str(raw_path)})
    except Exception as exc:
        n_synth_failed += 1
        tqdm.write(f"  Failed {row['id']}: {exc}")

print(f"Synthesised {len(SYNTH_ROWS)} of {len(PROMPT_ROWS)} prompt(s); {n_synth_failed} failed")
if not SYNTH_ROWS:
    raise RuntimeError("Donor synthesis produced no audio.")


---
## 3 · Post-Processing

For every synthesised clip: an optional `audiosronnx` super-resolution pass, resample to
`TARGET_SR`, downmix to mono, trim leading/trailing silence, and normalise loudness to
`TARGET_DBFS`. `SR_ENGINE` empty (the default) skips super-resolution entirely — plain resampling
is used. Every `audiosronnx` engine's constructor is read through `inspect.signature` before use:
only the options in `SR_<OPTION>` env vars that the selected engine actually accepts are read,
and setting one it does not accept stops the notebook, naming the engine and the option (the same
pattern [tts_enhance.ipynb](tts_enhance.ipynb) uses for its own `ENHANCE_<OPTION>`).

| Variable | Meaning |
|---|---|
| `TARGET_SR` | Final sample rate |
| `TRIM_SILENCE`, `TRIM_TOP_DB` | Silence trimming |
| `TARGET_DBFS` | Loudness normalisation target |
| `SR_ENGINE` | `audiosronnx` `load_sr()` engine alias; empty = off. `audiosronnx.available_models()` lists them |
| `SR_<OPTION>` | One env var per constructor option of the selected `SR_ENGINE` |
| `SR_CACHE_DIR`, `SR_REVISION` | Weight cache directory and Hugging Face revision pin |
| `ONNX_PROVIDERS` | onnxruntime execution providers for `SR_ENGINE`: comma list, or `auto` |


In [ ]:
%pip install --quiet --pre "librosa>=0.10.2" "soundfile>=0.12.1" "numpy>=1.26" "tqdm>=4.66"

sr_engine = _env("SR_ENGINE").strip().lower()
if sr_engine:
    %pip install --quiet --pre "audiosronnx>=0.9.0a2" "onnxruntime>=1.30.0"
else:
    print("SR_ENGINE not set — no super-resolution stage, plain resampling only.")


In [ ]:
import numpy as np
import soundfile as sf
import librosa

target_sr = int(_env("TARGET_SR", "24000"))
trim_silence = _flag("TRIM_SILENCE")
trim_top_db = int(_env("TRIM_TOP_DB", "30"))
target_dbfs = float(_env("TARGET_DBFS", "-20.0"))

sr_model = None
if sr_engine:
    import inspect
    from audiosronnx import load_sr
    from audiosronnx.base import get_engine

    def _parse_env_value(raw):
        low = raw.strip().lower()
        if low in ("true", "false"):
            return low == "true"
        for caster in (int, float):
            try:
                return caster(raw)
            except ValueError:
                pass
        return raw

    def _onnx_providers():
        raw = _env("ONNX_PROVIDERS", "auto").strip()
        if raw.lower() != "auto":
            return [p.strip() for p in raw.split(",") if p.strip()]
        import onnxruntime as ort
        priority = ["CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider", "CPUExecutionProvider"]
        available = set(ort.get_available_providers())
        ordered = [p for p in priority if p in available]
        if "CPUExecutionProvider" not in ordered:
            ordered.append("CPUExecutionProvider")
        return ordered

    _entry = get_engine(sr_engine)
    _sig = inspect.signature(_entry.adapter_class.__init__)
    _accepted = set(_sig.parameters) - {"self", "providers", "cache_dir", "revision"}
    sr_kwargs = {}
    for _name in _accepted:
        _raw = _env(f"SR_{_name.upper()}")
        if _raw:
            sr_kwargs[_name] = _parse_env_value(_raw)
    for _key in os.environ:
        if _key.startswith("SR_") and _key not in {"SR_ENGINE", "SR_CACHE_DIR", "SR_REVISION"}:
            _opt = _key[len("SR_"):].lower()
            if _opt not in _accepted:
                raise RuntimeError(
                    f"{_key} is set but {sr_engine!r} has no {_opt!r} option (accepts: {sorted(_accepted)})"
                )

    sr_model = load_sr(
        sr_engine,
        providers=_onnx_providers(),
        cache_dir=_env("SR_CACHE_DIR") or None,
        revision=_env("SR_REVISION") or None,
        **sr_kwargs,
    )
    print(f"Loaded audiosronnx SR engine {sr_engine!r} with options {sr_kwargs}")


wavs_dir = Path(_env("OUTPUT_DIR")) / "wavs"
wavs_dir.mkdir(parents=True, exist_ok=True)

FINAL_ROWS = []
n_post_failed = 0
for row in tqdm(SYNTH_ROWS, desc="Post-processing", unit="utt", file=sys.stdout):
    out_name = f"{_slug(row['id'])}.wav"
    out_path = wavs_dir / out_name
    try:
        wav, sr = librosa.load(row["raw_audio"], sr=None, mono=True)

        if sr_model is not None:
            wav, sr = sr_model.upscale(wav, sample_rate=sr)

        if sr != target_sr:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
            sr = target_sr

        if trim_silence:
            wav, _ = librosa.effects.trim(wav, top_db=trim_top_db)

        rms = float(np.sqrt(np.mean(wav ** 2) + 1e-9))
        current_dbfs = 20 * np.log10(rms + 1e-9)
        gain = 10 ** ((target_dbfs - current_dbfs) / 20.0)
        wav = np.clip(wav * gain, -1.0, 1.0).astype(np.float32)

        sf.write(str(out_path), wav, sr)
        FINAL_ROWS.append({
            **row,
            "audio": f"wavs/{out_name}",
            "duration_s": round(len(wav) / sr, 3),
            "sample_rate": int(sr),
        })
    except Exception as exc:
        n_post_failed += 1
        tqdm.write(f"  Failed to post-process {row['id']}: {exc}")

print(f"Post-processed {len(FINAL_ROWS)} of {len(SYNTH_ROWS)} clip(s); {n_post_failed} failed")
if not FINAL_ROWS:
    raise RuntimeError("Post-processing produced no audio.")


---
## 4 · Manifest, Sidecar and Summary


In [ ]:
import csv

manifest_rows = []
for row in FINAL_ROWS:
    manifest_rows.append({
        "id": row["id"],
        "text": row["text"],
        "lang": _env("TTS_LANG"),
        "audio": row["audio"],
        "duration_s": row["duration_s"],
        "sample_rate": row["sample_rate"],
        "source": {
            "kind": "tts",
            "plugin": DONOR_PLUGIN_ID,
            "voice": DONOR_VOICE_ID,
            "version": DONOR_VERSION,
        },
    })

out_dir = Path(_env("OUTPUT_DIR"))
manifest_path = out_dir / _env("MANIFEST_NAME", "manifest.jsonl")
with open(manifest_path, "w", encoding="utf-8") as f:
    for row in manifest_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

metadata_csv_path = out_dir / "metadata.csv"
with open(metadata_csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    for row in manifest_rows:
        writer.writerow([Path(row["audio"]).stem, row["text"]])

meta = {
    "notebooks": ["tts_dataset_gen.ipynb"],
    "prompts_source": {
        "kind": prompts_source_kind,
        "name": prompts_source_name,
        "path": os.path.basename(prompts_source_path) if prompts_source_path else None,
    },
    "donor": {
        "plugin": DONOR_PLUGIN_ID,
        "voice": DONOR_VOICE_ID,
        "version": DONOR_VERSION,
        "registry_admitted": DONOR_ADMITTED,
        "note": DONOR_NOTE,
    },
    "counts": {
        "prompt_lines_read": len(raw_lines),
        "prompts_refused": {k: v for k, v in reason_counts.items() if k != "passed"},
        "prompts_passed_text_validation": reason_counts.get("passed", 0),
        "donor_synthesis_failed": n_synth_failed,
        "post_processing_failed": n_post_failed,
        "rows_written": len(manifest_rows),
        "hours_written": round(sum(r["duration_s"] for r in manifest_rows) / 3600, 4),
    },
}
meta_path = out_dir / (_env("MANIFEST_NAME", "manifest.jsonl") + ".meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("=" * 65)
print("  OUTPUT SUMMARY")
print("=" * 65)
print(f"  Manifest     : {manifest_path}  ({len(manifest_rows)} rows)")
print(f"  Sidecar      : {meta_path}")
print(f"  metadata.csv : {metadata_csv_path}")
print(f"  Audio        : {wavs_dir}")
if manifest_rows:
    print("  First row    :", json.dumps(manifest_rows[0], ensure_ascii=False))
print("=" * 65)


---
## Next steps

This manifest feeds any of the later chain steps directly:

- [tts_voice_convert.ipynb](tts_voice_convert.ipynb) — turn this donor speech into a target voice identity
- [tts_measure.ipynb](tts_measure.ipynb) — score naturalness, speaker similarity and the recogniser round trip
- [tts_select.ipynb](tts_select.ipynb) — decide what a trainer keeps
- [tts_publish.ipynb](tts_publish.ipynb) — publish the selected rows as a Hugging Face dataset


Licensed under Apache-2.0.
